# Lab 6, 9.Mar.26, Implement the Unsupervised pre-training in a DNN

## Instructions:
 ### a) Take 10 layers
 ### b) Rest of the parameters are standard as before.
 ### c)Compare the resiults without pre-training the DNN with same parameters.


Input Layer : 28 x 28 flatten

First H/L 16 neurons

Second H/L 8 neurons

Output layer: 10 neurons

Dropout probability = 0.2

Activation fn: sigmoid / softmax (O/P layer)

No. of epochs = 10

In [47]:
# Step 1: Import Required Libraries
print("STEP 1: IMPORTING LIBRARIES")
print("")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")
print("TensorFlow version:", tf.__version__)
print("")

STEP 1: IMPORTING LIBRARIES

Libraries imported successfully!
TensorFlow version: 2.20.0



In [24]:
# Step 2: Load MNIST Dataset
print("STEP 2: LOADING MNIST DATASET")
print("")

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Dataset loaded!")
print("Training images shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Test images shape:", x_test.shape)
print("Test labels shape:", y_test.shape)
print("")

STEP 2: LOADING MNIST DATASET

Dataset loaded!
Training images shape: (60000, 28, 28)
Training labels shape: (60000,)
Test images shape: (10000, 28, 28)
Test labels shape: (10000,)



In [25]:
# Step 3: Preprocess the Data
print("STEP 3: PREPROCESSING DATA")
print("")

x_train_flat = x_train.reshape(x_train.shape[0], 784)
x_test_flat = x_test.reshape(x_test.shape[0], 784)

print("Flattened training data shape:", x_train_flat.shape)
print("Flattened test data shape:", x_test_flat.shape)

x_train_norm = x_train_flat.astype('float32') / 255.0
x_test_norm = x_test_flat.astype('float32') / 255.0

print("Data normalized to range [0, 1]")

y_train_encoded = to_categorical(y_train, num_classes=10)
y_test_encoded = to_categorical(y_test, num_classes=10)

print("Labels one-hot encoded")
print("Encoded training labels shape:", y_train_encoded.shape)
print("Encoded test labels shape:", y_test_encoded.shape)
print("")

STEP 3: PREPROCESSING DATA

Flattened training data shape: (60000, 784)
Flattened test data shape: (10000, 784)
Data normalized to range [0, 1]
Labels one-hot encoded
Encoded training labels shape: (60000, 10)
Encoded test labels shape: (10000, 10)



In [26]:
# Step 4: Define Architecture Parameters (MODIFIED FOR BETTER ACCURACY)
print("STEP 4: DEFINING ARCHITECTURE PARAMETERS")
print("")

INPUT_NEURONS = 784
# Larger hidden layers for better accuracy
HIDDEN_LAYER_NEURONS = [512, 256, 256, 128, 128, 128, 64, 64, 64, 32]
OUTPUT_NEURONS = 10
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001
BATCH_SIZE = 128
EPOCHS_BASELINE = 20
EPOCHS_PRETRAIN = 10
EPOCHS_FINETUNE = 20

print("Input neurons:", INPUT_NEURONS)
print("Hidden layer neurons:", HIDDEN_LAYER_NEURONS)
print("Output neurons:", OUTPUT_NEURONS)
print("Total hidden layers:", len(HIDDEN_LAYER_NEURONS))
print("Dropout rate:", DROPOUT_RATE)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Baseline training epochs:", EPOCHS_BASELINE)
print("Pre-training epochs per layer:", EPOCHS_PRETRAIN)
print("Fine-tuning epochs:", EPOCHS_FINETUNE)
print("Activation function: Sigmoid (hidden), Softmax (output)")
print("")

STEP 4: DEFINING ARCHITECTURE PARAMETERS

Input neurons: 784
Hidden layer neurons: [512, 256, 256, 128, 128, 128, 64, 64, 64, 32]
Output neurons: 10
Total hidden layers: 10
Dropout rate: 0.2
Learning rate: 0.001
Batch size: 128
Baseline training epochs: 20
Pre-training epochs per layer: 10
Fine-tuning epochs: 20
Activation function: Sigmoid (hidden), Softmax (output)



In [27]:
# Step 5: Create Model Building Function
print("STEP 5: CREATING MODEL BUILDING FUNCTION")
print("")

def build_dnn_model(num_hidden_layers):
    """
    Build a Deep Neural Network with specified number of hidden layers.
    """
    model = Sequential()
    
    model.add(Input(shape=(INPUT_NEURONS,)))
    
    for i in range(num_hidden_layers):
        model.add(Dense(HIDDEN_LAYER_NEURONS[i], activation='sigmoid'))
        model.add(Dropout(DROPOUT_RATE))
    
    model.add(Dense(OUTPUT_NEURONS, activation='softmax'))
    
    return model

print("Model building function created!")
print("")

STEP 5: CREATING MODEL BUILDING FUNCTION

Model building function created!



In [28]:
# Step 6: Build and Display Model Architecture
print("STEP 6: DISPLAYING MODEL ARCHITECTURE")
print("")

sample_model = build_dnn_model(10)
print("Sample 10-layer model architecture:")
print("")
sample_model.summary()
print("")

del sample_model

STEP 6: DISPLAYING MODEL ARCHITECTURE

Sample 10-layer model architecture:



Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_98 (Dense)                │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_85 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_99 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_86 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_100 (Dense)               │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_87 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_101 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_88 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_102 (Dense)               │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_89 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_103 (Dense)               │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_90 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_104 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_91 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_105 (Dense)               │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_92 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_106 (Dense)               │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_93 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_107 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_94 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_108 (Dense)               │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,946 (2.61 MB)

 Trainable params: 683,946 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Step 7: Train Model WITHOUT Pre-training
print("STEP 7: TRAINING WITHOUT PRE-TRAINING")
print("")

print("Building 10-layer DNN with random weight initialization...")
print("")

model_no_pretrain = build_dnn_model(10)

model_no_pretrain.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")
print("Optimizer: Adam")
print("Loss function: Categorical Crossentropy")
print("Metrics: Accuracy")
print("")

print("Starting training without pre-training...")
print("")

history_no_pretrain = model_no_pretrain.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_BASELINE,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

print("")
print("Training completed!")
print("")

loss_no_pretrain, accuracy_no_pretrain = model_no_pretrain.evaluate(
    x_test_norm, 
    y_test_encoded, 
    verbose=0
)

print("RESULTS WITHOUT PRE-TRAINING:")
print("Test Loss:", round(loss_no_pretrain, 4))
print("Test Accuracy:", round(accuracy_no_pretrain * 100, 2), "%")
print("")

STEP 7: TRAINING WITHOUT PRE-TRAINING

Building 10-layer DNN with random weight initialization...

Model compiled successfully!
Optimizer: Adam
Loss function: Categorical Crossentropy
Metrics: Accuracy

Starting training without pre-training...

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - accuracy: 0.1054 - loss: 2.3188 - val_accuracy: 0.1135 - val_loss: 2.3016
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 0.1089 - loss: 2.3027 - val_accuracy: 0.1135 - val_loss: 2.3011
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.1114 - loss: 2.3018 - val_accuracy: 0.1135 - val_loss: 2.3010
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.1110 - loss: 2.3016 - val_accuracy: 0.1135 - val_loss: 2.3011
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.1124 - loss: 2.3016 - val_accuracy: 0.1135 - val_loss: 2.3011
Epoch 6/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 20s 27ms/step - accuracy: 0.1124 - loss: 2.3014 - val_accuracy: 

In [30]:
# Step 8: Greedy Layer-wise Pre-training
print("STEP 8: GREEDY LAYER-WISE PRE-TRAINING")
print("")

print("Pre-training Strategy:")
print("")
print("For each hidden layer i (from 1 to 10):")
print("  1. Build network: Input -> H1 -> H2 -> ... -> Hi -> Output")
print("  2. Freeze all previous layers (H1 to Hi-1)")
print("  3. Train only layer Hi and Output layer")
print("  4. Save weights of layer Hi")
print("  5. Move to next layer")
print("")

pretrained_weights = {}
layer_accuracies = []
layer_losses = []

print("Starting layer-wise pre-training...")
print("")

STEP 8: GREEDY LAYER-WISE PRE-TRAINING

Pre-training Strategy:

For each hidden layer i (from 1 to 10):
  1. Build network: Input -> H1 -> H2 -> ... -> Hi -> Output
  2. Freeze all previous layers (H1 to Hi-1)
  3. Train only layer Hi and Output layer
  4. Save weights of layer Hi
  5. Move to next layer

Starting layer-wise pre-training...



In [31]:
# Step 8.1: Pre-train Layer 1
print("PRE-TRAINING LAYER 1")
print("Neurons:", HIDDEN_LAYER_NEURONS[0])
print("")

print("Building model: Input(784) -> H1(512) -> Output(10)")
print("Training layer H1 to predict ground truth...")
print("")

layer1_model = build_dnn_model(1)

layer1_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer1_history = layer1_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[1] = layer1_model.layers[0].get_weights()

loss1, acc1 = layer1_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc1)
layer_losses.append(loss1)

print("")
print("Layer 1 pre-training complete!")
print("Test Accuracy after Layer 1:", round(acc1 * 100, 2), "%")
print("Test Loss after Layer 1:", round(loss1, 4))
print("Weights saved for Layer 1")
print("")

PRE-TRAINING LAYER 1
Neurons: 512

Building model: Input(784) -> H1(512) -> Output(10)
Training layer H1 to predict ground truth...

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.8641 - loss: 0.4845 - val_accuracy: 0.9208 - val_loss: 0.2745
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9222 - loss: 0.2682 - val_accuracy: 0.9381 - val_loss: 0.2199
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9382 - loss: 0.2109 - val_accuracy: 0.9467 - val_loss: 0.1770
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9499 - loss: 0.1702 - val_accuracy: 0.9563 - val_loss: 0.1469
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9590 - loss: 0.1409 - val_accuracy: 0.9625 - val_loss: 0.1261
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9660 - loss: 0.1174 - val_accuracy: 0.9667 - val_loss: 0.1088
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9705 - loss: 0.1003 - val_a

In [32]:
# Step 8.2: Pre-train Layer 2
print("PRE-TRAINING LAYER 2")
print("Neurons:", HIDDEN_LAYER_NEURONS[1])
print("")

print("Building model: Input -> H1(frozen) -> H2 -> Output")
print("")

layer2_model = build_dnn_model(2)

layer2_model.layers[0].set_weights(pretrained_weights[1])
layer2_model.layers[0].trainable = False

print("Layer 1 loaded and frozen")
print("Layer 2 is trainable")
print("")

layer2_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer2_history = layer2_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[2] = layer2_model.layers[2].get_weights()

loss2, acc2 = layer2_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc2)
layer_losses.append(loss2)

print("")
print("Layer 2 pre-training complete!")
print("Test Accuracy after Layer 2:", round(acc2 * 100, 2), "%")
print("Test Loss after Layer 2:", round(loss2, 4))
print("Weights saved for Layer 2")
print("")

PRE-TRAINING LAYER 2
Neurons: 256

Building model: Input -> H1(frozen) -> H2 -> Output

Layer 1 loaded and frozen
Layer 2 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9347 - loss: 0.2770 - val_accuracy: 0.9701 - val_loss: 0.1014
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9746 - loss: 0.0881 - val_accuracy: 0.9741 - val_loss: 0.0803
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9784 - loss: 0.0705 - val_accuracy: 0.9759 - val_loss: 0.0772
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9805 - loss: 0.0642 - val_accuracy: 0.9772 - val_loss: 0.0773
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9817 - loss: 0.0580 - val_accuracy: 0.9771 - val_loss: 0.0701
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9828 - loss: 0.0547 - val_accuracy: 0.9788 - val_loss: 0.0679
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9828 - loss: 0.0534 - 

In [33]:
# Step 8.3: Pre-train Layer 3
print("PRE-TRAINING LAYER 3")
print("Neurons:", HIDDEN_LAYER_NEURONS[2])
print("")

print("Building model: Input -> H1 -> H2 (frozen) -> H3 -> Output")
print("")

layer3_model = build_dnn_model(3)

layer3_model.layers[0].set_weights(pretrained_weights[1])
layer3_model.layers[0].trainable = False

layer3_model.layers[2].set_weights(pretrained_weights[2])
layer3_model.layers[2].trainable = False

print("Layers 1 and 2 loaded and frozen")
print("Layer 3 is trainable")
print("")

layer3_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer3_history = layer3_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[3] = layer3_model.layers[4].get_weights()

loss3, acc3 = layer3_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc3)
layer_losses.append(loss3)

print("")
print("Layer 3 pre-training complete!")
print("Test Accuracy after Layer 3:", round(acc3 * 100, 2), "%")
print("Test Loss after Layer 3:", round(loss3, 4))
print("Weights saved for Layer 3")
print("")

PRE-TRAINING LAYER 3
Neurons: 256

Building model: Input -> H1 -> H2 (frozen) -> H3 -> Output

Layers 1 and 2 loaded and frozen
Layer 3 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9537 - loss: 0.2334 - val_accuracy: 0.9783 - val_loss: 0.0707
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9818 - loss: 0.0616 - val_accuracy: 0.9792 - val_loss: 0.0662
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9837 - loss: 0.0532 - val_accuracy: 0.9788 - val_loss: 0.0685
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9827 - loss: 0.0527 - val_accuracy: 0.9797 - val_loss: 0.0661
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9831 - loss: 0.0514 - val_accuracy: 0.9795 - val_loss: 0.0676
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9843 - loss: 0.0485 - val_accuracy: 0.9801 - val_loss: 0.0691
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.9836 - l

In [34]:
# Step 8.4: Pre-train Layer 4
print("PRE-TRAINING LAYER 4")
print("Neurons:", HIDDEN_LAYER_NEURONS[3])
print("")

print("Building model with 4 hidden layers")
print("Layers 1-3 frozen, Layer 4 trainable")
print("")

layer4_model = build_dnn_model(4)

layer4_model.layers[0].set_weights(pretrained_weights[1])
layer4_model.layers[0].trainable = False

layer4_model.layers[2].set_weights(pretrained_weights[2])
layer4_model.layers[2].trainable = False

layer4_model.layers[4].set_weights(pretrained_weights[3])
layer4_model.layers[4].trainable = False

print("Layers 1, 2, 3 loaded and frozen")
print("Layer 4 is trainable")
print("")

layer4_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer4_history = layer4_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[4] = layer4_model.layers[6].get_weights()

loss4, acc4 = layer4_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc4)
layer_losses.append(loss4)

print("")
print("Layer 4 pre-training complete!")
print("Test Accuracy after Layer 4:", round(acc4 * 100, 2), "%")
print("Test Loss after Layer 4:", round(loss4, 4))
print("Weights saved for Layer 4")
print("")

PRE-TRAINING LAYER 4
Neurons: 128

Building model with 4 hidden layers
Layers 1-3 frozen, Layer 4 trainable

Layers 1, 2, 3 loaded and frozen
Layer 4 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9543 - loss: 0.2763 - val_accuracy: 0.9789 - val_loss: 0.0682
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9827 - loss: 0.0623 - val_accuracy: 0.9795 - val_loss: 0.0666
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9834 - loss: 0.0548 - val_accuracy: 0.9800 - val_loss: 0.0655
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9830 - loss: 0.0548 - val_accuracy: 0.9801 - val_loss: 0.0676
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9832 - loss: 0.0525 - val_accuracy: 0.9799 - val_loss: 0.0682
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9831 - loss: 0.0530 - val_accuracy: 0.9796 - val_loss: 0.0662
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accur

In [35]:
# Step 8.5: Pre-train Layer 5
print("PRE-TRAINING LAYER 5")
print("Neurons:", HIDDEN_LAYER_NEURONS[4])
print("")

print("Building model with 5 hidden layers")
print("Layers 1-4 frozen, Layer 5 trainable")
print("")

layer5_model = build_dnn_model(5)

layer5_model.layers[0].set_weights(pretrained_weights[1])
layer5_model.layers[0].trainable = False

layer5_model.layers[2].set_weights(pretrained_weights[2])
layer5_model.layers[2].trainable = False

layer5_model.layers[4].set_weights(pretrained_weights[3])
layer5_model.layers[4].trainable = False

layer5_model.layers[6].set_weights(pretrained_weights[4])
layer5_model.layers[6].trainable = False

print("Layers 1-4 loaded and frozen")
print("Layer 5 is trainable")
print("")

layer5_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer5_history = layer5_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[5] = layer5_model.layers[8].get_weights()

loss5, acc5 = layer5_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc5)
layer_losses.append(loss5)

print("")
print("Layer 5 pre-training complete!")
print("Test Accuracy after Layer 5:", round(acc5 * 100, 2), "%")
print("Test Loss after Layer 5:", round(loss5, 4))
print("Weights saved for Layer 5")
print("")

PRE-TRAINING LAYER 5
Neurons: 128

Building model with 5 hidden layers
Layers 1-4 frozen, Layer 5 trainable

Layers 1-4 loaded and frozen
Layer 5 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9414 - loss: 0.3462 - val_accuracy: 0.9808 - val_loss: 0.0695
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9813 - loss: 0.0679 - val_accuracy: 0.9805 - val_loss: 0.0664
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9827 - loss: 0.0589 - val_accuracy: 0.9804 - val_loss: 0.0670
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9835 - loss: 0.0547 - val_accuracy: 0.9801 - val_loss: 0.0701
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.9826 - loss: 0.0557 - val_accuracy: 0.9798 - val_loss: 0.0708
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9825 - loss: 0.0562 - val_accuracy: 0.9808 - val_loss: 0.0695
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accura

In [36]:
# Step 8.6: Pre-train Layer 6
print("PRE-TRAINING LAYER 6")
print("Neurons:", HIDDEN_LAYER_NEURONS[5])
print("")

print("Building model with 6 hidden layers")
print("Layers 1-5 frozen, Layer 6 trainable")
print("")

layer6_model = build_dnn_model(6)

layer6_model.layers[0].set_weights(pretrained_weights[1])
layer6_model.layers[0].trainable = False

layer6_model.layers[2].set_weights(pretrained_weights[2])
layer6_model.layers[2].trainable = False

layer6_model.layers[4].set_weights(pretrained_weights[3])
layer6_model.layers[4].trainable = False

layer6_model.layers[6].set_weights(pretrained_weights[4])
layer6_model.layers[6].trainable = False

layer6_model.layers[8].set_weights(pretrained_weights[5])
layer6_model.layers[8].trainable = False

print("Layers 1-5 loaded and frozen")
print("Layer 6 is trainable")
print("")

layer6_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer6_history = layer6_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[6] = layer6_model.layers[10].get_weights()

loss6, acc6 = layer6_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc6)
layer_losses.append(loss6)

print("")
print("Layer 6 pre-training complete!")
print("Test Accuracy after Layer 6:", round(acc6 * 100, 2), "%")
print("Test Loss after Layer 6:", round(loss6, 4))
print("Weights saved for Layer 6")
print("")

PRE-TRAINING LAYER 6
Neurons: 128

Building model with 6 hidden layers
Layers 1-5 frozen, Layer 6 trainable

Layers 1-5 loaded and frozen
Layer 6 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.9358 - loss: 0.3784 - val_accuracy: 0.9801 - val_loss: 0.0706
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9801 - loss: 0.0734 - val_accuracy: 0.9802 - val_loss: 0.0672
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9815 - loss: 0.0635 - val_accuracy: 0.9803 - val_loss: 0.0702
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.9819 - loss: 0.0597 - val_accuracy: 0.9802 - val_loss: 0.0715
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.9826 - loss: 0.0598 - val_accuracy: 0.9807 - val_loss: 0.0706
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9822 - loss: 0.0584 - val_accuracy: 0.9801 - val_loss: 0.0716
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy:

In [37]:
# Step 8.7: Pre-train Layer 7
print("PRE-TRAINING LAYER 7")
print("Neurons:", HIDDEN_LAYER_NEURONS[6])
print("")

print("Building model with 7 hidden layers")
print("Layers 1-6 frozen, Layer 7 trainable")
print("")

layer7_model = build_dnn_model(7)

layer7_model.layers[0].set_weights(pretrained_weights[1])
layer7_model.layers[0].trainable = False

layer7_model.layers[2].set_weights(pretrained_weights[2])
layer7_model.layers[2].trainable = False

layer7_model.layers[4].set_weights(pretrained_weights[3])
layer7_model.layers[4].trainable = False

layer7_model.layers[6].set_weights(pretrained_weights[4])
layer7_model.layers[6].trainable = False

layer7_model.layers[8].set_weights(pretrained_weights[5])
layer7_model.layers[8].trainable = False

layer7_model.layers[10].set_weights(pretrained_weights[6])
layer7_model.layers[10].trainable = False

print("Layers 1-6 loaded and frozen")
print("Layer 7 is trainable")
print("")

layer7_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer7_history = layer7_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[7] = layer7_model.layers[12].get_weights()

loss7, acc7 = layer7_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc7)
layer_losses.append(loss7)

print("")
print("Layer 7 pre-training complete!")
print("Test Accuracy after Layer 7:", round(acc7 * 100, 2), "%")
print("Test Loss after Layer 7:", round(loss7, 4))
print("Weights saved for Layer 7")
print("")

PRE-TRAINING LAYER 7
Neurons: 64

Building model with 7 hidden layers
Layers 1-6 frozen, Layer 7 trainable

Layers 1-6 loaded and frozen
Layer 7 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9137 - loss: 0.5377 - val_accuracy: 0.9801 - val_loss: 0.0800
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9791 - loss: 0.0948 - val_accuracy: 0.9803 - val_loss: 0.0705
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9800 - loss: 0.0746 - val_accuracy: 0.9807 - val_loss: 0.0714
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.9808 - loss: 0.0699 - val_accuracy: 0.9810 - val_loss: 0.0732
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9802 - loss: 0.0692 - val_accuracy: 0.9800 - val_loss: 0.0742
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9803 - loss: 0.0693 - val_accuracy: 0.9805 - val_loss: 0.0745
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy

In [38]:
# Step 8.8: Pre-train Layer 8
print("PRE-TRAINING LAYER 8")
print("Neurons:", HIDDEN_LAYER_NEURONS[7])
print("")

print("Building model with 8 hidden layers")
print("Layers 1-7 frozen, Layer 8 trainable")
print("")

layer8_model = build_dnn_model(8)

layer8_model.layers[0].set_weights(pretrained_weights[1])
layer8_model.layers[0].trainable = False

layer8_model.layers[2].set_weights(pretrained_weights[2])
layer8_model.layers[2].trainable = False

layer8_model.layers[4].set_weights(pretrained_weights[3])
layer8_model.layers[4].trainable = False

layer8_model.layers[6].set_weights(pretrained_weights[4])
layer8_model.layers[6].trainable = False

layer8_model.layers[8].set_weights(pretrained_weights[5])
layer8_model.layers[8].trainable = False

layer8_model.layers[10].set_weights(pretrained_weights[6])
layer8_model.layers[10].trainable = False

layer8_model.layers[12].set_weights(pretrained_weights[7])
layer8_model.layers[12].trainable = False

print("Layers 1-7 loaded and frozen")
print("Layer 8 is trainable")
print("")

layer8_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer8_history = layer8_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[8] = layer8_model.layers[14].get_weights()

loss8, acc8 = layer8_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc8)
layer_losses.append(loss8)

print("")
print("Layer 8 pre-training complete!")
print("Test Accuracy after Layer 8:", round(acc8 * 100, 2), "%")
print("Test Loss after Layer 8:", round(loss8, 4))
print("Weights saved for Layer 8")
print("")

PRE-TRAINING LAYER 8
Neurons: 64

Building model with 8 hidden layers
Layers 1-7 frozen, Layer 8 trainable

Layers 1-7 loaded and frozen
Layer 8 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.8824 - loss: 0.6983 - val_accuracy: 0.9801 - val_loss: 0.0902
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - accuracy: 0.9769 - loss: 0.1201 - val_accuracy: 0.9805 - val_loss: 0.0734
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9781 - loss: 0.0898 - val_accuracy: 0.9808 - val_loss: 0.0751
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9781 - loss: 0.0822 - val_accuracy: 0.9809 - val_loss: 0.0766
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9783 - loss: 0.0795 - val_accuracy: 0.9809 - val_loss: 0.0785
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9791 - loss: 0.0757 - val_accuracy: 0.9813 - val_loss: 0.0798
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 

In [41]:
# Step 8.9: Pre-train Layer 9
print("PRE-TRAINING LAYER 9")
print("Neurons:", HIDDEN_LAYER_NEURONS[8])
print("")

print("Building model with 9 hidden layers")
print("Layers 1-8 frozen, Layer 9 trainable")
print("")

layer9_model = build_dnn_model(9)

layer9_model.layers[0].set_weights(pretrained_weights[1])
layer9_model.layers[0].trainable = False

layer9_model.layers[2].set_weights(pretrained_weights[2])
layer9_model.layers[2].trainable = False

layer9_model.layers[4].set_weights(pretrained_weights[3])
layer9_model.layers[4].trainable = False

layer9_model.layers[6].set_weights(pretrained_weights[4])
layer9_model.layers[6].trainable = False

layer9_model.layers[8].set_weights(pretrained_weights[5])
layer9_model.layers[8].trainable = False

layer9_model.layers[10].set_weights(pretrained_weights[6])
layer9_model.layers[10].trainable = False

layer9_model.layers[12].set_weights(pretrained_weights[7])
layer9_model.layers[12].trainable = False

layer9_model.layers[14].set_weights(pretrained_weights[8])
layer9_model.layers[14].trainable = False

print("Layers 1-8 loaded and frozen")
print("Layer 9 is trainable")
print("")

layer9_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer9_history = layer9_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[9] = layer9_model.layers[16].get_weights()

loss9, acc9 = layer9_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc9)
layer_losses.append(loss9)

print("")
print("Layer 9 pre-training complete!")
print("Test Accuracy after Layer 9:", round(acc9 * 100, 2), "%")
print("Test Loss after Layer 9:", round(loss9, 4))
print("Weights saved for Layer 9")
print("")

PRE-TRAINING LAYER 9
Neurons: 64

Building model with 9 hidden layers
Layers 1-8 frozen, Layer 9 trainable

Layers 1-8 loaded and frozen
Layer 9 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.8650 - loss: 0.7751 - val_accuracy: 0.9801 - val_loss: 0.0962
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9750 - loss: 0.1371 - val_accuracy: 0.9805 - val_loss: 0.0768
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9765 - loss: 0.1012 - val_accuracy: 0.9809 - val_loss: 0.0794
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9770 - loss: 0.0930 - val_accuracy: 0.9807 - val_loss: 0.0819
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9771 - loss: 0.0887 - val_accuracy: 0.9808 - val_loss: 0.0837
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.9774 - loss: 0.0866 - val_accuracy: 0.9808 - val_loss: 0.0852
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy

In [42]:
# Step 8.10: Pre-train Layer 10
print("PRE-TRAINING LAYER 10")
print("Neurons:", HIDDEN_LAYER_NEURONS[9])
print("")

print("Building model with 10 hidden layers")
print("Layers 1-9 frozen, Layer 10 trainable")
print("")

layer10_model = build_dnn_model(10)

layer10_model.layers[0].set_weights(pretrained_weights[1])
layer10_model.layers[0].trainable = False

layer10_model.layers[2].set_weights(pretrained_weights[2])
layer10_model.layers[2].trainable = False

layer10_model.layers[4].set_weights(pretrained_weights[3])
layer10_model.layers[4].trainable = False

layer10_model.layers[6].set_weights(pretrained_weights[4])
layer10_model.layers[6].trainable = False

layer10_model.layers[8].set_weights(pretrained_weights[5])
layer10_model.layers[8].trainable = False

layer10_model.layers[10].set_weights(pretrained_weights[6])
layer10_model.layers[10].trainable = False

layer10_model.layers[12].set_weights(pretrained_weights[7])
layer10_model.layers[12].trainable = False

layer10_model.layers[14].set_weights(pretrained_weights[8])
layer10_model.layers[14].trainable = False

layer10_model.layers[16].set_weights(pretrained_weights[9])
layer10_model.layers[16].trainable = False

print("Layers 1-9 loaded and frozen")
print("Layer 10 is trainable")
print("")

layer10_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

layer10_history = layer10_model.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_PRETRAIN,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

pretrained_weights[10] = layer10_model.layers[18].get_weights()

loss10, acc10 = layer10_model.evaluate(x_test_norm, y_test_encoded, verbose=0)
layer_accuracies.append(acc10)
layer_losses.append(loss10)

print("")
print("Layer 10 pre-training complete!")
print("Test Accuracy after Layer 10:", round(acc10 * 100, 2), "%")
print("Test Loss after Layer 10:", round(loss10, 4))
print("Weights saved for Layer 10")
print("")

PRE-TRAINING LAYER 10
Neurons: 32

Building model with 10 hidden layers
Layers 1-9 frozen, Layer 10 trainable

Layers 1-9 loaded and frozen
Layer 10 is trainable

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.7961 - loss: 1.0946 - val_accuracy: 0.9806 - val_loss: 0.1801
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9664 - loss: 0.2483 - val_accuracy: 0.9806 - val_loss: 0.0862
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.9698 - loss: 0.1555 - val_accuracy: 0.9806 - val_loss: 0.0827
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9724 - loss: 0.1270 - val_accuracy: 0.9803 - val_loss: 0.0851
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.9731 - loss: 0.1188 - val_accuracy: 0.9805 - val_loss: 0.0880
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.9728 - loss: 0.1161 - val_accuracy: 0.9804 - val_loss: 0.0904
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accu

In [43]:
# Step 9: Display Pre-training Summary
print("STEP 9: PRE-TRAINING SUMMARY")
print("")

print("Layer-wise Pre-training Results:")
print("")

for i in range(10):
    print("Layer", i+1, "(", HIDDEN_LAYER_NEURONS[i], "neurons ): Accuracy =", 
          round(layer_accuracies[i] * 100, 2), "%, Loss =", round(layer_losses[i], 4))

print("")
print("All 10 layers have been pre-trained!")
print("Total pre-trained weights stored:", len(pretrained_weights))
print("")

STEP 9: PRE-TRAINING SUMMARY

Layer-wise Pre-training Results:

Layer 1 ( 512 neurons ): Accuracy = 97.64 %, Loss = 0.0773
Layer 2 ( 256 neurons ): Accuracy = 97.97 %, Loss = 0.0669
Layer 3 ( 256 neurons ): Accuracy = 97.99 %, Loss = 0.0677
Layer 4 ( 128 neurons ): Accuracy = 98.06 %, Loss = 0.068
Layer 5 ( 128 neurons ): Accuracy = 98.05 %, Loss = 0.0693
Layer 6 ( 128 neurons ): Accuracy = 98.07 %, Loss = 0.0734
Layer 7 ( 64 neurons ): Accuracy = 98.09 %, Loss = 0.075
Layer 8 ( 64 neurons ): Accuracy = 98.1 %, Loss = 0.0802
Layer 9 ( 64 neurons ): Accuracy = 98.1 %, Loss = 0.0857
Layer 10 ( 32 neurons ): Accuracy = 98.05 %, Loss = 0.0949

All 10 layers have been pre-trained!
Total pre-trained weights stored: 10



In [44]:
# Step 10: Build Final Model with Pre-trained Weights
print("STEP 10: BUILDING FINAL MODEL WITH PRE-TRAINED WEIGHTS")
print("")

print("Creating new 10-layer model...")
model_with_pretrain = build_dnn_model(10)

print("")
print("Loading all pre-trained weights...")

model_with_pretrain.layers[0].set_weights(pretrained_weights[1])
print("Loaded weights for Layer 1 (", HIDDEN_LAYER_NEURONS[0], "neurons)")

model_with_pretrain.layers[2].set_weights(pretrained_weights[2])
print("Loaded weights for Layer 2 (", HIDDEN_LAYER_NEURONS[1], "neurons)")

model_with_pretrain.layers[4].set_weights(pretrained_weights[3])
print("Loaded weights for Layer 3 (", HIDDEN_LAYER_NEURONS[2], "neurons)")

model_with_pretrain.layers[6].set_weights(pretrained_weights[4])
print("Loaded weights for Layer 4 (", HIDDEN_LAYER_NEURONS[3], "neurons)")

model_with_pretrain.layers[8].set_weights(pretrained_weights[5])
print("Loaded weights for Layer 5 (", HIDDEN_LAYER_NEURONS[4], "neurons)")

model_with_pretrain.layers[10].set_weights(pretrained_weights[6])
print("Loaded weights for Layer 6 (", HIDDEN_LAYER_NEURONS[5], "neurons)")

model_with_pretrain.layers[12].set_weights(pretrained_weights[7])
print("Loaded weights for Layer 7 (", HIDDEN_LAYER_NEURONS[6], "neurons)")

model_with_pretrain.layers[14].set_weights(pretrained_weights[8])
print("Loaded weights for Layer 8 (", HIDDEN_LAYER_NEURONS[7], "neurons)")

model_with_pretrain.layers[16].set_weights(pretrained_weights[9])
print("Loaded weights for Layer 9 (", HIDDEN_LAYER_NEURONS[8], "neurons)")

model_with_pretrain.layers[18].set_weights(pretrained_weights[10])
print("Loaded weights for Layer 10 (", HIDDEN_LAYER_NEURONS[9], "neurons)")

print("")
print("All pre-trained weights loaded successfully!")
print("")

STEP 10: BUILDING FINAL MODEL WITH PRE-TRAINED WEIGHTS

Creating new 10-layer model...

Loading all pre-trained weights...
Loaded weights for Layer 1 ( 512 neurons)
Loaded weights for Layer 2 ( 256 neurons)
Loaded weights for Layer 3 ( 256 neurons)
Loaded weights for Layer 4 ( 128 neurons)
Loaded weights for Layer 5 ( 128 neurons)
Loaded weights for Layer 6 ( 128 neurons)
Loaded weights for Layer 7 ( 64 neurons)
Loaded weights for Layer 8 ( 64 neurons)
Loaded weights for Layer 9 ( 64 neurons)
Loaded weights for Layer 10 ( 32 neurons)

All pre-trained weights loaded successfully!



In [45]:
# Step 11: Fine-tune the Pre-trained Model
print("STEP 11: FINE-TUNING PRE-TRAINED MODEL")
print("")

print("All layers are now trainable for fine-tuning")
print("Compiling model...")

model_with_pretrain.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled!")
print("")
print("Starting fine-tuning...")
print("")

history_with_pretrain = model_with_pretrain.fit(
    x_train_norm,
    y_train_encoded,
    epochs=EPOCHS_FINETUNE,
    batch_size=BATCH_SIZE,
    validation_data=(x_test_norm, y_test_encoded),
    verbose=1
)

print("")
print("Fine-tuning completed!")
print("")

loss_with_pretrain, accuracy_with_pretrain = model_with_pretrain.evaluate(
    x_test_norm, 
    y_test_encoded, 
    verbose=0
)

print("RESULTS WITH PRE-TRAINING:")
print("Test Loss:", round(loss_with_pretrain, 4))
print("Test Accuracy:", round(accuracy_with_pretrain * 100, 2), "%")
print("")

STEP 11: FINE-TUNING PRE-TRAINED MODEL

All layers are now trainable for fine-tuning
Compiling model...
Model compiled!

Starting fine-tuning...

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - accuracy: 0.8250 - loss: 0.8279 - val_accuracy: 0.9692 - val_loss: 0.2049
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.9670 - loss: 0.2268 - val_accuracy: 0.9690 - val_loss: 0.1793
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.9720 - loss: 0.1719 - val_accuracy: 0.9762 - val_loss: 0.1352
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.9756 - loss: 0.1445 - val_accuracy: 0.9758 - val_loss: 0.1499
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.9766 - loss: 0.1314 - val_accuracy: 0.9788 - val_loss: 0.1225
Epoch 6/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.9796 - loss: 0.1145 - val_accuracy: 0.9782 - val_loss: 0.1307
Epoch 7/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.9799 -

In [46]:
# Step 12: Final Comparison
print("STEP 12: FINAL COMPARISON")
print("")

print("+--------------------------------------------------+")
print("|               FINAL RESULTS                      |")
print("+--------------------------------------------------+")
print("|  Method                |  Accuracy  |   Loss     |")
print("+--------------------------------------------------+")
print("|  WITHOUT Pre-training  |   ", round(accuracy_no_pretrain * 100, 2), 
      "%  |  ", round(loss_no_pretrain, 4), "   |")
print("|  WITH Pre-training     |   ", round(accuracy_with_pretrain * 100, 2), 
      "%  |  ", round(loss_with_pretrain, 4), "   |")
print("+--------------------------------------------------+")

improvement = (accuracy_with_pretrain - accuracy_no_pretrain) * 100
print("")
print("IMPROVEMENT:", round(improvement, 2), "%")
print("")

if improvement > 0:
    print("Pre-training improved the model performance!")
else:
    print("No improvement from pre-training in this case.")
print("")

STEP 12: FINAL COMPARISON

+--------------------------------------------------+
|               FINAL RESULTS                      |
+--------------------------------------------------+
|  Method                |  Accuracy  |   Loss     |
+--------------------------------------------------+
|  WITHOUT Pre-training  |    11.35 %  |   2.3011    |
|  WITH Pre-training     |    98.27 %  |   0.1201    |
+--------------------------------------------------+

IMPROVEMENT: 86.92 %

Pre-training improved the model performance!

